# Sprint 12 - Automatización de Reportes (Sesiones)
**Versión para estudiantes**

El presente caso tiene como objetivo que aprendas sobre el uso de herramientas complementarias a Python para la generación de reportes dinámicos y automatizables. Por un lado, se busca profundizar en los conocimientos asociados a la librería **plotly** y sus diversas funcionalidades para crear interactividad en gráficos; y por otro presentar a la aplicación **Quarto** que te ayudará a llevar tus resultados a páginas web, reportes, monitores, presentaciones o documentos pdf utilizando tus habilidades de programación en jupyter notebooks.

Sobre esta última idea, conviene que descargues la interfaz de comandos de **Quarto** desde el siguiente link: 

https://quarto.org/docs/download/

Y una vez que lo tengas, puedes trabajar con esta herramienta a través de la siguiente extensión en VS Code:

![](quarto.png)

Adicionalmente, te recomiendo que crees una cuenta en **QuatroPub** (https://quartopub.com/), un repositorio en el que podrás cargar y publicar tus archivos *quarto*, y compartirlos con quien tu desees. 

## Entendimiento del contexto

El sector de restaurantes ha experimentado una fuerte digitalización en los últimos años, impulsada por la necesidad de eficiencia operativa, personalización del servicio y toma de decisiones basada en datos. La adopción de sistemas de información tipo POS (*Point of Sale*) es una realidad propia de la industria, lo que ha permitdo consolidar datos en tiempo real sobre ventas, costos, comportamiento de clientes y desempeño del personal. Esta información es clave para el desarrollo de análisis operativos y estratégicos que requieren de profesionales capaces de transformar esta información en inteligencia de negocios. 

Justamente tú trabajas como analista de datos en un restaurante de estilo europeo que ofrece sus servicios en horarios de desayuno, almuerzo y cena. En una sesión de planificación, el dueño te ha pedido generar un reporte automatizable para el seguimiento de sus ventas. Este reporte además deberá ser compartido a todos los miembros del área administrativa por lo que es fundamental que sea lo más concreto y claro posible.

Luego de realizar un levantamiento de los requerimientos funcionales con los usuarios, defines que el reporte deberá responder las siguientes preguntas de negocio:

* Sobre las ventas: 
    * ¿Cuál es el nivel de ventas en el último año?
    * ¿Cómo se comparan estas ventas con los años previos?
* Sobre la demanda:
    * ¿Cómo se ha comportado el ticket promedio a través de los años?
    * ¿Cómo se comporta la demanda a través de los días de la semana?
    * ¿Existen diferencias en las preferencias de consumo durante la semana?
* Sobre los clientes:
    * ¿Existen clientes que regresan al restaurante? 
    * ¿Con que frecuencia regresan los clientes repetidores al restaurante?
* Sobre precios publicados:
    * ¿Cómo se distribuyen los precios de los artículos consumidos?

## Entendimiento de los datos

Antes de iniciar, carga las librerías con las que vas a trabajar. Además de **pandas** y **numpy**, importa el grupo `graph_objects` de la librería **plotly** que te permitirá desarrollar gráficos interactivos con un alto grado de control sobre sus componentes visuales.

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


El sistema POS que el restaurante maneja es uno llamado *Pixel*, y a partir de aquí tu extraes la información contenida en el archivo **registro_pixel**. En esta tabla encuentras transacciones de ventas a nivel de comanda, cliente y artículo entre 2018 y 2024, y con las siguientes columnas:

* comanda_id: Identificador único de la comanda en Pixel.
* cliente_id: Identificador único del cliente (DNI, Pasaporte).
* tipo_articulo: Categorización del artículo consumido.
* periodo: Año en el que se realizó la comanda.
* mes: Mes en el que se realizó la comanda.
* dia_semana: Día de la semana en el que se realizó la comanda (Lun = 1).
* valor_venta: Valor de la venta realizada en USD.
* valor cantidad: Cantidad consumida del tipo de artículo en la comanda.

Explora el dataset y a partir de aquí establece un plan de acción para preparar los datos.

In [3]:
df_pixel = pd.read_table('registro_pixel.txt', sep='\t')

In [4]:
df_pixel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113163 entries, 0 to 113162
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   comanda_id      113163 non-null  object 
 1   cliente_id      113142 non-null  object 
 2   tipo_articulo   113163 non-null  object 
 3   periodo         113163 non-null  int64  
 4   mes             113163 non-null  int64  
 5   dia_semana      113163 non-null  int64  
 6   valor_venta     113163 non-null  float64
 7   valor_cantidad  113163 non-null  float64
dtypes: float64(2), int64(3), object(3)
memory usage: 6.9+ MB


In [5]:
df_pixel.sample(10)

,comanda_id,cliente_id,tipo_articulo,periodo,mes,dia_semana,valor_venta,valor_cantidad
65355,cno_22574,1754111407,PLATOS FUERTES,2018,4,3,9.015,1.0
112698,cno_99343,1717158545001,BEBIDAS ALCOHOLICAS,2021,11,3,8.200,1.0
29276,cno_138267,4785LK,BEBIDAS ALCOHOLICAS,2023,5,3,19.680,2.0
85276,cno_60363,1702727924,BEBIDAS ALCOHOLICAS,2019,8,3,11.890,1.0
67887,cno_26069,0603250465,PLATOS FUERTES,2018,6,3,50.815,5.0
90633,cno_68923,0915527535,BEBIDAS ALCOHOLICAS,2019,12,5,21.320,2.0
38889,cno_152587,9321654,ENTRADAS,2023,12,5,12.300,1.0
108796,cno_94822,091112636,BEBIDAS ALCOHOLICAS,2021,8,7,10.660,1.0
35461,cno_148242,1302262421,BEBIDAS ALCOHOLICAS,2023,10,7,21.320,2.0
25752,cno_133209,179055776001,PLATOS FUERTES,2023,3,2,63.110,3.0


In [6]:
df_pixel['tipo_articulo'].unique()

array(['BEBIDAS CALIENTES', 'BEBIDAS FRIAS', 'TAPAS', 'DESAYUNOS',
       'BEBIDAS ALCOHOLICAS', 'ENTRADAS', 'PLATOS FUERTES', 'MENU',
       'POSTRES', 'SANDWICHES'], dtype=object)

In [7]:
df_pixel.duplicated(subset = ['comanda_id', 'tipo_articulo']).sum()

np.int64(14072)

**PLAN DE ACCIÓN PARA PREPARACIÓN DE DATOS**

- Existen pocos valores perdidos en la columna cliente_id, y dada su cantidad respecto al total de registros se podrían eliminar del dataset.
- La columna valor_cantidad representa un conteo de artículos por lo que su tipo debería ser entero y habrá que cambiarlo.
- Existen valores duplicados que deben eliminarse.

## Preparación de datos

Implementa tu plan de acción a fin de preparar los datos para el análisis.

In [8]:
# Eliminar registros con valores perdidos en cliente_id
df_pixel = df_pixel.dropna(subset = "cliente_id").reset_index(drop = True)
df_pixel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113142 entries, 0 to 113141
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   comanda_id      113142 non-null  object 
 1   cliente_id      113142 non-null  object 
 2   tipo_articulo   113142 non-null  object 
 3   periodo         113142 non-null  int64  
 4   mes             113142 non-null  int64  
 5   dia_semana      113142 non-null  int64  
 6   valor_venta     113142 non-null  float64
 7   valor_cantidad  113142 non-null  float64
dtypes: float64(2), int64(3), object(3)
memory usage: 6.9+ MB


In [9]:
# Cambiar tipo de columna cantidad
df_pixel["valor_cantidad"] = df_pixel["valor_cantidad"].astype(int)
df_pixel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113142 entries, 0 to 113141
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   comanda_id      113142 non-null  object 
 1   cliente_id      113142 non-null  object 
 2   tipo_articulo   113142 non-null  object 
 3   periodo         113142 non-null  int64  
 4   mes             113142 non-null  int64  
 5   dia_semana      113142 non-null  int64  
 6   valor_venta     113142 non-null  float64
 7   valor_cantidad  113142 non-null  int64  
dtypes: float64(1), int64(4), object(3)
memory usage: 6.9+ MB


In [10]:
# Verificar cantidades enteras
df_pixel["valor_cantidad"].describe().round(1)

count    113142.0
mean          3.0
std           4.4
min           0.0
25%           1.0
50%           2.0
75%           3.0
max         420.0
Name: valor_cantidad, dtype: float64

In [11]:
# Eliminar filas en las que las cantidades sean 0
df_pixel = df_pixel.query("valor_cantidad > 0")
df_pixel.shape

(112644, 8)

In [12]:
# Eliminar valores duplicados
df_pixel = df_pixel.drop_duplicates(subset = ["comanda_id", "cliente_id", "tipo_articulo"]).reset_index(drop = True)
df_pixel.duplicated(subset = ["comanda_id", "cliente_id", "tipo_articulo"]).sum()

np.int64(0)

## Análisis de datos

Como ya se mencionó vamos a utilizar la librería **plotly** para la realización de visualizaciones interactivas. Esta librería se caracteriza por su variedad y versatilidad en cuanto a componentes que se pueden adicionar a un gráfico. 

En general, para utilizar esta librería conviene que conozcas cómo se estructura un gráfico en **plotly** en sus objetos visuales:

![](go.png){width=1000}

* Figura: Corresponde al espacio en el que se va a generar la visualización y cuyos elementos fundamentales son las trazas. Usualmente viene representado por sistemas de coordenadas (cartesianas, polares, etc.).
* Trazas: Son todas las formas geométricas que se dibujan dentro de la figura. Conocemos varias como los puntos, las lineas o las barras. 
* Layout: Aquí se incluyen a todos los objetos que rodean a una figura y que facilitan su entendimiento. Los más comunes son los títulos, los nombres y ticks de los ejes, y las leyendas. 

Dado esto, a continuacion verás la sintaxis básica de todo gráfico realizado en **plotly** con Python:

```py
# Definicion de la FIGURA
fig = go.Figura(
    
    # Definicion de la TRAZA
    data = go.Trace(
        # Argumentos de la traza
    )
)

# Definicion del LAYOUT
fig.update_layout(
    # Argumentos del layout
)

fig.show()
```

Si gustas saber más de lo que puedes hacer con esta librería y los difrentes argumentos de cada objeto visual, te recomiendo mirar su documentación para Python en https://plotly.com/python/.

Entonces, para iniciar y a fin de estandarizar las visualizaciones que se generen en nuestro análisis, conviene parametrizar algunas características que desearíamos mantener. Por tanto guarda una paleta de colores y los formatos deseados para etiquetas (*hovers*), ejes y leyendas.

In [13]:
#Definir paleta de colores
color_fondo = "#dff0ef" #Codigo hexadecimal
color_linea = "#424B54"
color_fill = [
    "#081b27",
    "#16425b",
    "#3a7ca5",
    "#81c3d7",
    "#acddec",
    "#f45b4a",
    "#f79489",
    "#f8AFA6",
    "#fadcd9",
    "#f9f1f0"
]

In [14]:
# Definir formato de etiquetas
form_etiquetas = dict(
    font = dict(size = 15)
)

# Definir formato de ejes
form_ticks = dict(
    tickfont = dict(size = 15)
)

# Definir formato de leyendas
form_leyendas = dict(
    orientation = "h", 
    y = 1, 
    yanchor = "bottom",
    font = dict(size = 15)
)

### ¿Cuál es el nivel de ventas en el último año?

Crea una tabla interactiva que contenga las ventas del último año, incluye además información sobre la cantidad de comandas y clientes.

In [15]:
# Crear tabla con informacion requerida
res_vta = (
    df_pixel
    .groupby("periodo")
    .agg(
        ventas = ("valor_venta","sum"),
        comandas = ("comanda_id","nunique"),
        clientes = ("cliente_id","nunique")
    )
    .reset_index()
)

pref = res_vta["periodo"].max() 
res_vta = (
    res_vta
    .query("periodo == @pref")
    .sort_values(by = "periodo", ascending = False)
)

res_vta["ventas"] = res_vta["ventas"] / 1000
res_vta.columns = ["Año","Ventas","Comandas","Clientes"]

res_vta

,Año,Ventas,Comandas,Clientes
6,2024,560.706442,5216,4225


In [16]:
# Generar visualización interactiva
fig = go.Figure(
    
    data = go.Table(

        header = dict(
            values = res_vta.columns,
            fill_color = color_fill[1],
            font_color = color_fondo,
            font_size = 18
        ),
        
        cells = dict(
            values = res_vta.transpose(),
            format = ["",",.0f"],
            fill_color = color_fondo,
            height = 25,
            font_size = 18,
            font_color = color_fill[1]
        )
    )
)

fig.update_layout(
    autosize = False,
    width = 1000,
    height = 100,
    margin = dict(t = 20, b = 20)
)

fig.show()

### ¿Cómo se comparan estas ventas con los años previos?

Crea un gráfico interactivo de barras que muestre la evolución anual de ventas. En las etiquetas (*hovers*) debería mostrarse información asociada a la variación de este indicador.

In [17]:
# Crear tabla con información requerida
evol_vta = (
    df_pixel
    .groupby("periodo")
    .agg(ventas = ("valor_venta","sum"))
    .reset_index()
)

evol_vta["var"] = evol_vta["ventas"] / evol_vta["ventas"].shift(1) - 1

def fun_info (x):
    per = x["periodo"]
    vta = x["ventas"]/1000
    var = x["var"]
    if np.isnan(var):
        texto = f"<b>{per:.0f}</b><br>Ventas (KUSD): {vta:,.0f}"
    else:
        texto = f"<b>{per:.0f}</b><br>Ventas (KUSD): {vta:,.0f}<br>Var. YoY: {var:.1%}"
    return texto
evol_vta["info"] = evol_vta.apply(fun_info, axis = 1)

evol_vta

,periodo,ventas,var,info
0,2018,420709.804634,NaN,<b>2018</b><br>Ventas (KUSD): 421
1,2019,409980.208029,-0.025504,<b>2019</b><br>Ventas (KUSD): 410<br>Var. YoY:...
2,2020,212564.475199,-0.481525,<b>2020</b><br>Ventas (KUSD): 213<br>Var. YoY:...
3,2021,385329.799941,0.812767,<b>2021</b><br>Ventas (KUSD): 385<br>Var. YoY:...
4,2022,565217.318935,0.466840,<b>2022</b><br>Ventas (KUSD): 565<br>Var. YoY:...
5,2023,543743.858169,-0.037992,<b>2023</b><br>Ventas (KUSD): 544<br>Var. YoY:...
6,2024,560706.442177,0.031196,<b>2024</b><br>Ventas (KUSD): 561<br>Var. YoY:...


In [18]:
# Generar visualización interactiva
fig = go.Figure(
    data = go.Bar(

            x = evol_vta['periodo'],
            y = evol_vta["ventas"],
            
            hovertext = evol_vta["info"],
            hoverinfo = "text",
            hoverlabel = form_etiquetas,
            
            opacity = 0.9,
            width = 0.5,
            marker = dict(color = color_fill[1])

        )
)

fig.update_layout(
    plot_bgcolor = color_fondo,
    xaxis = form_ticks,
    yaxis = form_ticks,
    autosize = False,
    width = 1000,
    height = 500,
    margin=dict(t = 40)
)

fig.show()

### ¿Cómo se ha comportado el ticket promedio a través de los años?

Crea un gráfico interactivo de líneas que muestre la evolución del ticket promedio (ventas / comandas). En las etiquetas debería mostrarse información asociada a la variación de este indicador. 

In [19]:
# Crear tabla con información requerida
evol_tkt = (
    df_pixel
    .groupby("periodo")
    .agg(
        ventas = ("valor_venta","sum"),
        ordenes = ("comanda_id","nunique"),
    )
    .reset_index()
)

evol_tkt["ticket"] = evol_tkt["ventas"]/evol_tkt["ordenes"]
evol_tkt["var"] = evol_tkt["ticket"] / evol_tkt["ticket"].shift(1) - 1

def fun_info (x):
    per = x["periodo"]
    tik = x["ticket"]
    var = x["var"]
    if np.isnan(var):
        texto = f"<b>{per:.0f}</b><br>Ticket (USD): {tik:,.1f}"
    else:
        texto = f"<b>{per:.0f}</b><br>Ticket (USD): {tik:,.1f}<br>Var. YoY: {var:.1%}"
    return texto
evol_tkt["info"] = evol_tkt.apply(fun_info, axis = 1)

evol_tkt

,periodo,ventas,ordenes,ticket,var,info
0,2018,420709.804634,4466,94.202822,NaN,<b>2018</b><br>Ticket (USD): 94.2
1,2019,409980.208029,4523,90.643424,-0.037784,<b>2019</b><br>Ticket (USD): 90.6<br>Var. YoY:...
2,2020,212564.475199,2607,81.536047,-0.100475,<b>2020</b><br>Ticket (USD): 81.5<br>Var. YoY:...
3,2021,385329.799941,4308,89.445172,0.097002,<b>2021</b><br>Ticket (USD): 89.4<br>Var. YoY:...
4,2022,565217.318935,5421,104.264401,0.165679,<b>2022</b><br>Ticket (USD): 104.3<br>Var. YoY...
5,2023,543743.858169,5030,108.100171,0.036789,<b>2023</b><br>Ticket (USD): 108.1<br>Var. YoY...
6,2024,560706.442177,5216,107.497401,-0.005576,<b>2024</b><br>Ticket (USD): 107.5<br>Var. YoY...


In [20]:
# Generar visualización interactiva
fig = go.Figure(
    data = go.Scatter(

        x = evol_tkt['periodo'],
        y = evol_tkt["ticket"],
        
        mode = "lines+markers",
        
        hovertext = evol_tkt["info"],
        hoverinfo = "text",
        hoverlabel = form_etiquetas,
        
        marker = dict(
            color = color_fill[5],  
            size = 10
        ),
        
        line = dict(
            color = color_linea, 
            width = 3, 
            shape = "spline", 
            smoothing = 1, 
            dash = "dot"
        )
    )
)

fig.update_layout(
    plot_bgcolor = color_fondo,
    xaxis = form_ticks,
    yaxis = form_ticks,
    autosize = False,
    width = 1000,
    height = 600
)

fig.show()

### ¿Cómo se comporta la demanda a través de los días de la semana?

Crea un gráfico interactivo de barras conexas que muestre para cada día de la semana y para cada año el nivel promedio de comandas. Excluye los años de pandemia (2020 y 2021) puesto que no son representativos en el análisis. 

In [21]:
# Crear tabla con informacion requerida
com_diaper = (
    df_pixel
    .query("periodo != 2020 and periodo != 2021")
    .pivot_table(
        index = "dia_semana",
        columns = "periodo",
        values = "comanda_id",
        aggfunc = "nunique"
    )
)
com_diaper.index = ["Lun","Mar","Mie","Jue","Vie","Sab","Dom"]

com_diaper

periodo,2018,2019,2022,2023,2024
Lun,485,533,641,592,580
Mar,509,531,621,608,590
Mie,915,628,668,584,559
Jue,559,631,737,641,628
Vie,661,700,837,720,827
Sab,663,741,1010,929,1084
Dom,674,759,907,956,948


In [22]:
# Generar gráfico interactivo
i = len(com_diaper.columns) - 1

fig = go.Figure()

for col in com_diaper.columns:
    
    etiqueta = [f"{col}: {x:,.0f}" for x in com_diaper[col]] 

    fig.add_trace(
        
        go.Bar(
            
            name = col,
            x = com_diaper.index,
            y = com_diaper[col],
            
            hovertext = etiqueta,
            hoverinfo = "text",
            hoverlabel = form_etiquetas,
            
            opacity = 0.9,
            marker = dict(color = color_fill[i])
            
        )
    )

    i -= 1

fig.update_layout(
    plot_bgcolor = color_fondo,
    xaxis = form_ticks,
    yaxis = form_ticks,
    legend = form_leyendas,
    autosize = False,
    width = 1000,
    height = 600,
    margin = dict(t = 100)
)

fig.show()

### ¿Existen diferencias en las preferencias de consumo durante la semana?

Crea un gráfico interactivo de barras apiladas que muestre cómo se distribuyen los tipos de artículo consumidos por día de la semana. Excluye nuevamente los años de pandemia puesto que no son representativos.

In [23]:
# Crear tabla con informacion requerida
art_dia = (
    df_pixel
    .query("periodo != 2020 and periodo != 2021")
    .pivot_table(
        index = "dia_semana",
        columns = "tipo_articulo",
        values = "valor_cantidad",
        aggfunc = "sum"
    )
)
art_dia.index = ["Lun","Mar","Mie","Jue","Vie","Sab","Dom"]

art_dia = art_dia.apply(lambda x: 100*x/x.sum(), axis = 1)

art_dia

tipo_articulo,BEBIDAS ALCOHOLICAS,BEBIDAS CALIENTES,BEBIDAS FRIAS,DESAYUNOS,ENTRADAS,MENU,PLATOS FUERTES,POSTRES,SANDWICHES,TAPAS
Lun,10.920489,6.407361,28.629209,5.133304,19.107486,6.430955,15.538104,5.645623,0.519060,1.668408
Mar,10.815977,6.429607,28.361455,4.702742,18.441867,8.888083,14.605853,5.427762,0.438307,1.888347
Mie,10.862320,6.688711,27.590606,2.996772,15.923766,11.515830,15.736305,6.548115,0.554572,1.583004
Jue,11.794260,6.159698,28.714979,5.061171,18.467740,7.346220,14.357490,5.668625,0.491073,1.938744
Vie,10.653675,6.777665,28.095722,5.763156,18.825954,6.456448,14.591252,6.394882,0.500562,1.940682
Sab,9.961840,5.844547,26.810381,11.963580,17.707706,5.279954,14.219723,5.998527,0.435161,1.778581
Dom,8.647054,5.328509,23.351260,21.128157,15.038787,7.829190,11.814419,4.904706,0.413889,1.544028


In [24]:
# Generar visualizacion interactiva
i = 0

fig = go.Figure()

for col in art_dia.columns:
    
    etiqueta = [f"{col}: {x:.1f}%" for x in art_dia[col]] 

    fig.add_trace(

        go.Bar(
            
            name = col,
            x = art_dia.index,
            y = art_dia[col],
            
            hovertext = etiqueta,
            hoverinfo = "text",
            hoverlabel = form_etiquetas,
            
            opacity = 0.9,
            width = 0.8,
            marker = dict(
                color = color_fill[i],
                line = dict(color = color_linea))
        )
    )
    i += 1

form_ticks_y = form_ticks.copy()
form_ticks_y["range"] = [0,100]
form_leyendas_c = form_leyendas.copy()
form_leyendas_c["font"] = dict(size = 10)
fig.update_layout(
    barmode = "stack",
    plot_bgcolor = color_fondo,
    xaxis = form_ticks,
    yaxis = form_ticks_y,
    legend = form_leyendas_c,
    autosize = False,
    width = 1000,
    height = 600,
    margin = dict(t = 100)
)

fig.show()

### ¿Existen clientes que regresan al restaurante?

Crea un gráfico interactivo tipo pastel que muestre el porcentaje de clientes que han tenido más de una comanda en todo el periodo de estudio.

In [25]:
# Crear tabla con información requerida
com_cli = (
    df_pixel
    .groupby("cliente_id")
    .agg(
        comandas = ("comanda_id","nunique")
    )
    .reset_index()
)

com_cli["tipo"] = com_cli["comandas"].apply(lambda x: "Repetidor" if x > 1 else "De una ocasion")

cli_tipo = (
    com_cli
    .groupby("tipo")
    .agg(
        casos = ("cliente_id","nunique")
    )
    .reset_index()
)

total_clientes = cli_tipo["casos"].sum()

def fun_info (x):
    grp = x["tipo"]
    n = x["casos"]
    prc = n/total_clientes
    texto = f"<b>{grp}</b><br>Casos: {n:,.0f}<br>Part.: {prc:.1%}"
    return texto
cli_tipo["info"] = cli_tipo.apply(fun_info, axis = 1)

cli_tipo

,tipo,casos,info
0,De una ocasion,19962,"<b>De una ocasion</b><br>Casos: 19,962<br>Part..."
1,Repetidor,2577,"<b>Repetidor</b><br>Casos: 2,577<br>Part.: 11.4%"


In [26]:
# Generar visualización interactiva
fig = go.Figure(
    data = go.Pie(
        
        values = cli_tipo["casos"],
        labels = cli_tipo["tipo"],
        
        hovertext = cli_tipo["info"],
        hoverinfo = "text",
        hoverlabel = form_etiquetas,
        
        marker = dict(colors = [color_fill[1], color_fill[5]])

    )
)

fig.update_layout(
    showlegend = False,
    width = 400
)

fig.show()

### ¿Con que frecuencia regresan los clientes repetidores al restaurante?

Crea un gráfico interactivo con barras horizontales que muestre un conteo de clientes repetidores por su frecuencia de repetición. Por frecuencia en este contexto hacemos referencia a los siguientes tipos:

* Clientes fieles: Aquellos que vienen al restaurante al menos una vez al semestre.
* Clientes habituales: Aquellos que vienen al restaurante una vez cada dos años.
* Clientes esporádicos: El resto de casos de repetidores.   

In [27]:
# Crear tabla simple con información requerida
cli_repf = (
    df_pixel.
    groupby("cliente_id")
    .agg(
        comandas = ("comanda_id","nunique")
    )
    .reset_index()
    .query("comandas > 1")
)

umbral = df_pixel["periodo"].max() - df_pixel["periodo"].min()
def fun_tipo_frec (x):
    if x > umbral:
        return "a.Fiel"
    elif umbral > x >= umbral/2:
        return "b.Habitual"
    else:
        return "c.Esporadico"    
cli_repf["tipo"] = cli_repf["comandas"].apply(fun_tipo_frec)

cli_repf = cli_repf["tipo"].value_counts().sort_index().reset_index()

def fun_info(x):
    tip = x["tipo"]
    n = x["count"]
    return f"<b>{tip}</b><br>Casos: {n:,.0f}"
cli_repf["info"] = cli_repf.apply(fun_info, axis = 1)

cli_repf

,tipo,count,info
0,a.Fiel,162,<b>a.Fiel</b><br>Casos: 162
1,b.Habitual,621,<b>b.Habitual</b><br>Casos: 621
2,c.Esporadico,1794,"<b>c.Esporadico</b><br>Casos: 1,794"


In [28]:
# Generar visualizacion interactiva
fig = go.Figure(
    data = go.Bar(
        
        y = cli_repf["tipo"],
        x = cli_repf["count"],
        
        hovertext = cli_repf["info"],
        hoverinfo = "text",
        hoverlabel = form_etiquetas,
        
        orientation = "h",
        marker = dict(color = color_fill[6])
    )
)

fig.update_layout(
    plot_bgcolor=color_fondo,
    xaxis = form_ticks,
    yaxis = form_ticks,
    width = 400
)

fig.show()

### ¿Cómo se distribuyen los precios de los artículos consumidos?

Crea un gráfico interactivo en el cual se muestre la distribución de los precios pagados por artículos tipo comida (entradas, platos fuertes, postres y menúes). Te recomiendo utilizar una traza llamada violín en donde excluyas los valores atípicos utilizando un criterio *2-sigma*.

In [29]:
# Crear tabla con información requerida
articulos = ["PLATOS FUERTES","ENTRADAS","POSTRES","MENU"]
precio_comida = (
    df_pixel
    .query("tipo_articulo in @articulos")
    .groupby(["comanda_id","tipo_articulo"])
    .agg(
        venta = ("valor_venta","sum"),
        cantidad = ("valor_cantidad","sum")
    )
    .reset_index()
)

precio_comida["precio"] = precio_comida["venta"] / precio_comida["cantidad"]

umbral = precio_comida["precio"].mean() + 2*precio_comida["precio"].std()
precio_comida = precio_comida.query("precio <= @umbral")

precio_comida.sample(5)

,comanda_id,tipo_articulo,venta,cantidad,precio
12427,cno_143697,POSTRES,4.5100,1,4.5100
28215,cno_32646,MENU,63.9400,2,31.9700
5924,cno_119490,ENTRADAS,22.9600,2,11.4800
17127,cno_159100,ENTRADAS,36.9000,15,2.4600
40623,cno_87991,PLATOS FUERTES,12.7855,1,12.7855


In [30]:
# Generar visualizacion interactiva
fig = go.Figure(
    data = go.Violin(
        
        x = precio_comida["tipo_articulo"],
        y = precio_comida["precio"],
        
        hoveron = "kde",
        hoverlabel = form_etiquetas,

        opacity = 0.9,
        points = False,
        bandwidth = 1,
        meanline = dict(
            visible = True, 
            color = color_fill[5]
        ),
        marker = dict(color = color_fill[1])
    )
)

fig.update_layout(
    plot_bgcolor = color_fondo,
    xaxis = form_ticks,
    yaxis = form_ticks,
    autosize = False,
    width = 1000,
    height = 600
)

fig.show()

Crea ahora el mismo tipo de gráfico interactivo pero para los artículos tipo bebida.

In [31]:
# Crear tabla con información requerida
articulos = ["BEBIDAS CALIENTES", "BEBIDAS FRIAS", "BEBIDAS ALCOHOLICAS"]
precio_bebida = (
    df_pixel
    .query("tipo_articulo in @articulos")
    .groupby(["comanda_id","tipo_articulo"])
    .agg(
        venta = ("valor_venta","sum"),
        cantidad = ("valor_cantidad","sum")
    )
    .reset_index()
)

precio_bebida["precio"] = precio_bebida["venta"] / precio_bebida["cantidad"]

umbral = precio_bebida["precio"].mean() + 2*precio_bebida["precio"].std()
precio_bebida = precio_bebida.query("precio <= @umbral")

precio_bebida.sample(5)

,comanda_id,tipo_articulo,venta,cantidad,precio
37224,cno_74208,BEBIDAS FRIAS,29.930,9,3.325556
43260,cno_95714,BEBIDAS CALIENTES,3.280,1,3.280000
14500,cno_150021,BEBIDAS FRIAS,18.860,6,3.143333
41202,cno_88585,BEBIDAS ALCOHOLICAS,10.127,1,10.127000
36462,cno_71270,BEBIDAS ALCOHOLICAS,10.650,1,10.650000


In [32]:
# Generar visualizacion interactiva
fig = go.Figure(
    data = go.Violin(
        
        x = precio_bebida["tipo_articulo"],
        y = precio_bebida["precio"],
        
        hoveron = "kde",
        hoverlabel = form_etiquetas,

        opacity = 0.9,
        points = False,
        bandwidth = 1,
        meanline = dict(
            visible = True, 
            color = color_fill[5]
        ),
        marker = dict(color = color_fill[1])
    )
)

fig.update_layout(
    plot_bgcolor = color_fondo,
    xaxis = form_ticks,
    yaxis = form_ticks,
    autosize = False,
    width = 1000,
    height = 600
)

fig.show()

## Comunicación de resultados

**Quarto** es una sistema de publicación libre enfocado en comunicar resultados derivados del análisis de datos, y permitiendo crear servicios web, dashboards, artículos o presentaciones de acuerdo a la necesidad de sus usuarios. **Quarto** presenta algunas ventajas importantes a destacar: 

* La forma de construir publicaciones en **Quarto** es bastante similar a la de un Jupyter Notebook en cuanto a estructura. En ambos casos se combinan espacios markdown (texto) y celdas de código.
* La compilación y despliegue de publicaciones se lo hace localmente, lo cual agiliza significativamente los tiempos con respecto a otras herramientas como **Render**.
* Existe un control casi total del usuario, no solamente del contenido de la publicación, sino de sus aspectos formales. 

Entonces, a continuación aprenderemos sobre los aspectos básicos vinculados a esta herramienta, llevando las visualizaciones creadas en el apartado anterior a un reporte tipo presentación. Sin embargo, considerando que el alcance de lo que se muestra aquí es limitado, te sugiero que revises todas las alternativas y potencialidades de **Quarto** en el siguiente link:

https://quarto.org/docs/guide/

Antes de empezar además, te sugiero que ya tengas activa una cuenta en **QuartoPub** pues es allí donde podremos publicar nuestros archivos *quarto (qmd)*.

**TAREA 1:** Crea un nuevo archivo tipo *Quarto document* en VS Code, y guárdalo en tu directorio de trabajo activo.

### Cabecera qmd

Una vez creado el archivo notarás que el mismo viene predefinido con un bloque de texto entre "---". A esta parte se la conoce como **cabecera** e incluirá toda la información y características principales de tu archivo, como su título y el tipo de publicación deseada (servicio web, presentación, artículo, etc.).

![](cabecera.png)

**TAREA 2:** Cambia entonces la cabecera de tu archivo por la siguiente:

```
---
title: "REPORTE DE VENTAS"
format: revealjs
author: "DA XX"
jupyter: python3
date: last-modified
---
```

Ahora bien, vale aclarar algunas cosas:

* El argumento `format: revealjs` le indica a **Quarto** que el archivo sera de tipo presentación con gráficos interactivos. 
* El argumento `author: "DA XX"` permite especificar el nombre de quien crea el archivo, por lo que en este caso deberías sustituir "XX" por tu cohorte.
* El argumento `jupyter: python3` indica a la herramienta que el archivo contedrá celdas con código Python a ser ejecutado.
* El argumento `date: last-modified` hará que en la publicación siempre aparezca la fecha en la que se generó la última versión de la misma. 

Veamos de manera preliminar como quedaría publicada nuestra presentación. 

**TAREA 3:** Guarda el archivo con los cambios realizados en la cabecera y has click en el icono ![](preview.png) (o tambien con *Ctrl + Shift + K*).

Notarás que se inicializa una terminal y se ejecuta el siguiente comando

```ps
quarto preview [ruta del archivo qmd]
```

Luego de un momento en VS Code se abrirá una ventana donde puedes ver cómo está quedando tu publicación:

![](ventana_preview.png)

Por ahora nuestra presentación contiene solamente una diapositiva con el título y el autor. Vamos a ir incorporando nuevas a continuación.

### Creación de diapositivas

Para crear nuevas diapositivas debes trabajar en la parte de tu archivo que está fuera de los separadores "---". Una vez allí recuerda cómo definir títulos y subtitulos en las celdas **markdown** de los jupyter notebooks, tal que:

* Para láminas tipo sección utiliza `# Nombre de la sección`.
* Para láminas simples utiliza `## Nombre de la lámina`.

**TAREA 4:** Crea entonces láminas nuevas en tu archivo en base a los resultados alcanzados de tu análisis. Para esto incorpora el siguiente texto:

```
# ANÁLISIS DE VENTAS

## Evolucion de ventas

# ANÁLISIS DE DEMANDA

## Comportamiento del ticket

## Comportamiento de comandas

## Comportamiento por tipo de consumo

# ANÁLISIS DE CLIENTES

## Grados de repetición de clientes

# ANÁLISIS DE PRECIOS

## Distribución de precios de comida

## Districuión de precios de bebidas
```

**TAREA 5:** Una vez hecho esto, vuelve a generar una vista preliminar de tu presentación para verificar que la misma está funcionando adecuadamente.

### Creación de celdas con código Python

Empecemos a incorporar nuestro código Python en celdas específicas del archivo. 

**TAREA 6:** Antes del primer título incorpora lo siguiente:

![](py_cell.png)

Esto se conoce como una **celda de código** y dentro de ella puedes ingresar el código Python que necesites. 

**TAREA 7:** Prueba entonces añadiendo aquí lo siguiente desde tu análisis:

* Carga de librerías
* Carga de datos
* Preparacion de datos
* Parametrización de visualizaciones

No incluyas las partes en las que muestras o imprimes algún resultado.

**TAREA 8:** Ahora que ya sabes como hacerlo, en cada diapositiva de las secciones de "Análisis de ventas", "Análisis de demanda" y "Análisis de precios", crea celdas en la que incorpores los códigos Python correspondientes de tu análisis. Ten en cuenta que ahora solamente queremos ver en nuestro reporte las visualizaciones finales según corresponda.

**TAREA 9:** Para la lámina "Grados de repetición de clientes" de la sección "Análisis de clientes", incorpora el siguiente contenido:

```
::: {.columns}

::: {.column width="50%"}

**Participación por tipo**

:::

::: {.column width="50%"}

**Participación por frecuencia**

:::

:::
```

Con esto le estamos indicando a **Quarto** que deseamos mostrar dos columnas de igual tamaño en esta diapositiva. 

**TAREA 10:** A continuación de los nombres de cada columna ingresa las celdas de código Python que correspondan y genera una vista preliminar de tu presentación a fin de verificar que todo el contenido se encuentre de manera correcta.

### Mejoramiento del formato

Ya tenemos una presentación interactiva con todos los resultados de nuestro análisis. Finalicemos entonces con el archivo dándole un mejor formato que haga destacar nuestras habilidades comunicacionales.

**TAREA 11:** Dado que estamos hablando de una presentación, hagamos que nuestras diapositivas sigan un mismo tema y que transicionen mediante una animación. Para esto modifica el atributo `format` de la **cabecera** lo siguiente:

```
format: 
    revealjs:
        theme: simple
        transition: fade
```

Trabajemos ahora con los colores de nuestra presentación. Recuerda que luego de cada cambio puedes generar una vista preliminar del archivo para verificar que todo esté trabajando adecuadamente.

**TAREA 12:** Incorpora en la cabecera los siguientes atributos:

```
title-slide-attributes:
    data-background-color: "#16425b"
```

**TAREA 13:** Ahora adiciona el siguiente texto luego de los nombres de cada sección

```
{background-color="#3a7ca5"}
```

**TAREA 14:** Para concluir incluye una diapositiva vacía tipo sección al final de la presentación, la cual tenga el mismo color de fondo que la diapositiva principal. Una vez hecho esto guarda tu archivo. 

### Publicación de presentación

**TAREA 15:** Ingresa a la terminal de comandos y verifica que estés ubicado en el directorio de trabajo donde se ha guardado tu archivo *qmd*. Una vez allí escribe el siguiente comando:

```ps
quarto publish [nombre del archivo qmd]
```

A partir de allí, sigue con estos pasos:

* Selecciona como destino a publicar **QuartoPub** (ya debes tener una cuenta creada).
* Especifica tu usuario de **QuartoPub**.
* Especifica un nombre para tu publicación.
* Espera a que el archivo se compile y se publique.

¡Listo! Si todo ha funcionado correctamente ya cuentas con un reporte publicado tipo presentación interactiva. Desde el sitio de **QuartoPub** podrás extraer el link de tu reporte para copartirlo con quien tu desees, que en este caso en particular deberían ser los administradores del restaurante.

**NOTA**: Si algún momento deseas realizar alguna modificación a este o cualquier publicación de **Quarto**, simplemente hasla en el archivo *qmd* correspondiente y vuelve a ejecutar el comando de publicación. De forma automática **Quarto** te preguntará si deseas hacer una actualización de la publicación ya existente, ante lo cual tu deberías aceptar.